# Agent Framework + AgentMemory Integration
## Automatic Memory Management Demo

## 🎯 Objective
Demonstrate how **AgentMemory** automatically integrates with **Agent Framework** through the `ContextProvider` interface for seamless multi-session memory management without manual intervention.

## Key Concepts Explained

### 1. **ContextProvider Pattern**
- Pass memory as `context_providers=[memory]` to Agent
- No manual `add_turn()` or `store_response()` calls needed
- System automatically manages all memory lifecycle

### 2. **Automatic Context Injection (before_run)**
- When agent processes query, `before_run()` is called automatically
- Loads previous conversation context and long-term profile
- Agent receives enriched context, enabling personalization
- Example: "Based on our Session 1 discussion about risk..."

### 3. **Automatic Turn Capture (after_run)**
- After agent responds, `after_run()` is called automatically
- Response stored in memory for future sessions
- User's question + Agent's response recorded
- No developer code needed

### 4. **Multi-Session Learning**
- **Session 1**: Builds initial user profile
- **Session 2**: Recalls Session 1 context, adds new insights
- **Session 3**: Uses accumulated knowledge from Sessions 1+2
- Profile evolves with each session

## Demo Scenario: Financial Advisor

A financial advisor AI helps user **Sarah** (35, software engineer, $150k salary) across 3 sessions:

```
Session 1: Initial Consultation
  → Sarah introduces herself
  → Discusses risk tolerance (moderate-to-high)
  → Asks about 401k strategy
  → System extracts insights: age, income, horizon, risk profile

Session 2: Investment Strategy (2 weeks later)
  → Agent: "Hi Sarah! Based on our last conversation..."
  → Recalls: She's 35, $150k, moderate-to-high risk, 30-year horizon
  → Recommends asset allocation (85% stocks, 15% bonds)
  → System updates profile with investment preferences

Session 3: Tax Planning (1 month later)
  → Agent: "Given your situation we discussed..."
  → References: Income level, account types, asset allocation, goals
  → Provides tax-optimized strategy using ALL accumulated knowledge
  → Profile now comprehensive and highly personalized
```

## What Makes This Powerful

✅ **No Manual Code**: Developer doesn't manage memory lifecycle  
✅ **Seamless Context**: Agent naturally references previous conversations  
✅ **True Learning**: Agent personality and recommendations evolve with user  
✅ **Insight Extraction**: System automatically identifies key facts about user  
✅ **Production Ready**: Works with SQLite (local) or CosmosDB (cloud) backends  

---

Let's walk through the implementation step by step!

In [1]:
# ============================================================================
# STEP 1-6: Setup, Configuration & Agent Initialization
# ============================================================================

print("\n" + "="*80)
print("INITIALIZATION: Environment Setup & Agent Creation")
print("="*80)

import asyncio
import os
import sys
import time
from pathlib import Path

# Setup UTF-8 encoding for Windows
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# ========== STEP 1: Setup Paths & Load Environment ==========
print("\n🔧 STEP 1: Setup paths and load environment variables")
print("-" * 80)

BASE_DIR = Path.cwd()
print(f"📁 Working directory: {BASE_DIR}")

# Find project root
while BASE_DIR != BASE_DIR.parent:
    if (BASE_DIR / "pyproject.toml").exists():
        print(f"✅ Found project root: {BASE_DIR}")
        break
    BASE_DIR = BASE_DIR.parent

sys.path.insert(0, str(BASE_DIR))

# Load environment variables
from dotenv import load_dotenv
env_path = BASE_DIR / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loaded .env from: {env_path}")
else:
    print(f"⚠️  .env file not found at: {env_path}")

print("✅ Paths configured")

# ========== STEP 2: Validate Azure Configuration ==========
print("\n🔐 STEP 2: Validate Azure OpenAI configuration")
print("-" * 80)

required_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_REASONING_MODEL",
    "AZURE_OPENAI_EMB_DEPLOYMENT"
]

print("Checking required environment variables:")
missing = []
for var in required_vars:
    value = os.getenv(var)
    if not value:
        missing.append(var)
        print(f"   ❌ {var}: NOT SET")
    else:
        # Show truncated for sensitive values
        if "KEY" in var or "ENDPOINT" in var:
            display = value[:20] + "..." if len(value) > 20 else value
        else:
            display = value
        print(f"   ✅ {var}: {display}")

if missing:
    print(f"\n⚠️  ERROR: Missing configuration: {', '.join(missing)}")
    print("   Please check your .env file")
else:
    print("\n✅ All Azure configuration validated")

# Demo configuration
USER_ID = "sarah_demo"
DB_PATH = str(BASE_DIR / "demo_financial_advisor.db")
print(f"\n📊 Demo Configuration:")
print(f"   • User ID: {USER_ID}")
print(f"   • Database: {DB_PATH}")

# Clean up previous demo (with Windows retry handling)
if os.path.exists(DB_PATH):
    print(f"   • Cleaning previous database...")
    for attempt in range(5):
        try:
            os.remove(DB_PATH)
            print(f"     ✅ Cleaned up previous database")
            break
        except PermissionError:
            if attempt < 4:
                time.sleep(0.5)
            else:
                print(f"     ⚠️  Could not delete (file in use)")

# ========== STEP 3: Import Required Libraries ==========
print("\n📦 STEP 3: Import required libraries")
print("-" * 80)

try:
    from azure.identity import DefaultAzureCredential
    from openai import AzureOpenAI
    print("✅ Azure SDK imported")
    
    from agent_framework import Agent
    from agent_framework.azure import AzureOpenAIChatClient
    print("✅ Agent Framework imported")
    
    from memory import AgentMemory, AgentMemoryConfig
    print("✅ AgentMemory imported")
except ImportError as e:
    print(f"❌ Import error: {e}")
    raise

# ========== STEP 4: Define Agent Tools ==========
print("\n🛠️  STEP 4: Define financial advisor tools")
print("-" * 80)
print("Creating lightweight functions that agent can call:")

def get_401k_limit(year: int = 2025) -> str:
    """Get 401k contribution limits for a given year."""
    limits = {
        2024: "$23,000 (under 50), $30,500 (50+)",
        2025: "$23,500 (under 50), $31,000 (50+)"
    }
    return limits.get(year, "Information not available")

def get_roth_ira_limit(year: int = 2025) -> str:
    """Get Roth IRA contribution limits."""
    return "$7,000 (under 50), $8,000 (50+)"

print("   ✅ get_401k_limit(year) - Returns 401k contribution limits")
print("   ✅ get_roth_ira_limit(year) - Returns Roth IRA contribution limits")
print("   (Agent can call these tools during conversations)")

# ========== STEP 5: Initialize Azure OpenAI Client ==========
print("\n🤖 STEP 5: Initialize Azure OpenAI client")
print("-" * 80)

try:
    openai_client = AzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    )
    print("✅ Azure OpenAI client created")
except Exception as e:
    print(f"❌ Error initializing OpenAI client: {e}")
    raise

# ========== STEP 5b: Create AgentMemory Configuration ==========
print("\n⚙️  Configuring AgentMemory...")
print("   Key settings:")
print("   • auto_enrich_context=True: Auto-inject previous context")
print("   • enrichment_trigger_keywords: Detect when user references past")
print("   • longterm_synthesis_frequency=1: Update profile after each session")

config = AgentMemoryConfig(
    # Automatic context enrichment with keyword triggers
    auto_enrich_context=True,
    enrichment_trigger_keywords=[
        "remember", "recall", "previous", "last time", "before",
        "discussed", "mentioned", "told you", "my profile",
        "based on", "given my", "considering my"
    ],
    # Synthesize long-term insights after every session
    longterm_synthesis_frequency=1,
    # Don't auto-manage sessions (we'll do it manually)
    auto_manage_sessions=False,
)
print("✅ AgentMemory configuration created")

# Initialize AgentMemory
print("   Initializing memory store...")
memory = AgentMemory(
    user_id=USER_ID,
    openai_client=openai_client,
    db_path=DB_PATH,
    config=config,
)
print(f"✅ AgentMemory initialized (storing at {DB_PATH})")

# ========== STEP 6: Create Agent with Memory Integration ==========
print("\n🧠 STEP 6: Create Agent with automatic memory integration")
print("-" * 80)
print("This is the KEY integration point!")
print()

# Setup Azure credentials
try:
    credential = DefaultAzureCredential()
    print("✅ Azure credentials configured")
except Exception as e:
    print(f"⚠️  Credential warning: {e}")

# Create chat client
print("Creating chat client...")
try:
    chat_client = AzureOpenAIChatClient(
        credential=credential,
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        deployment_name=os.getenv("AZURE_OPENAI_REASONING_MODEL", "gpt-4o")
    )
    print("✅ Chat client created")
except Exception as e:
    print(f"❌ Error creating chat client: {e}")
    raise

print("\nCreating Agent with AUTOMATIC MEMORY INTEGRATION...")
print("   Key line: context_providers=[memory]")
print("   Effect: Agent Framework automatically:")
print("     • Calls memory.before_run() BEFORE each agent turn")
print("     • Calls memory.after_run() AFTER each agent turn")
print()

agent = Agent(
    client=chat_client,
    instructions="""You are an expert financial advisor specializing in retirement planning.

Your approach:
- Provide personalized advice based on client's profile
- Reference details from previous conversations when relevant
- Explain complex concepts clearly
- Be proactive about suggesting strategies

Always be professional, accurate, and personalized.""",
    tools=[get_401k_limit, get_roth_ira_limit],
    context_providers=[memory],  # ← THE MAGIC: Automatic memory management!
)

print("✅ Agent created with automatic memory integration")
print("\n" + "="*80)
print("✅ INITIALIZATION COMPLETE - Agent ready for conversations!")
print("="*80)


INITIALIZATION: Environment Setup & Agent Creation

🔧 STEP 1: Setup paths and load environment variables
--------------------------------------------------------------------------------
📁 Working directory: c:\LabFiles\agent-memory\demo
✅ Found project root: c:\LabFiles\agent-memory
✅ Loaded .env from: c:\LabFiles\agent-memory\.env
✅ Paths configured

🔐 STEP 2: Validate Azure OpenAI configuration
--------------------------------------------------------------------------------
Checking required environment variables:
   ✅ AZURE_OPENAI_ENDPOINT: https://openai-23196...
   ✅ AZURE_OPENAI_API_KEY: 7LC2kRlisRQZEqRunw2i...
   ✅ AZURE_OPENAI_REASONING_MODEL: gpt-5.1
   ✅ AZURE_OPENAI_EMB_DEPLOYMENT: text-embedding-3-large

✅ All Azure configuration validated

📊 Demo Configuration:
   • User ID: sarah_demo
   • Database: c:\LabFiles\agent-memory\demo_financial_advisor.db

📦 STEP 3: Import required libraries
--------------------------------------------------------------------------------
✅ Azu

## Step 7-8: Run Three-Session Demo

### What Happens in Each Session

**Session 1 - Building the Profile**
- Sarah introduces herself (35, engineer, $150k)
- Discusses her risk tolerance and time horizon
- Asks about 401k strategy
- **Memory captures**: Demographics, risk tolerance, goals
- **System creates**: Initial user profile

**Session 2 - Automatic Context Recall**
- User asks about asset allocation
- **Agent automatically receives**: Sarah's age, income, risk tolerance, 30-year horizon
- Agent says: "Based on our previous discussion..."
- **Memory updates**: Investment preferences, strategy alignment

**Session 3 - Using Accumulated Knowledge**
- User asks about tax optimization
- **Agent automatically receives**: ALL accumulated context from Sessions 1+2
- Agent provides highly personalized tax strategy
- **Memory refines**: Tax preferences, complete financial picture

### How to Read the Output

- `📚 Memory context loaded (X chars)` → Previous context was injected
- `Turn N:` → This is turn number N in the conversation
- `Session ended (X turns)` → This session processed X user queries
- `💡 Insights extracted: N` → System learned N new facts about the user

In [2]:
# ============================================================================
# STEP 7-8: Run Three-Session Demo
# ============================================================================

print("\n" + "="*80)
print("MULTI-SESSION DEMO: Automatic Memory Management in Action")
print("="*80)

# Helper function for running sessions
async def run_session(agent, memory, queries, session_name):
    """Run a conversation session with automatic memory management.
    
    Memory lifecycle:
    1. start_session() → Opens new session
    2. get_context() → Retrieves previous context (if any)
    3. For each query:
       - agent.run() calls memory.before_run() internally
       - Agent receives enriched context automatically
       - Agent responds
       - agent.run() calls memory.after_run() internally
       - Response stored in memory automatically
    4. end_session() → Closes session, triggers insight extraction
    """
    print(f"\n{'='*80}")
    print(f"SESSION: {session_name}")
    print(f"{'='*80}")
    
    # Start session
    await memory.start_session()
    session_id = memory.session_id[:8]
    print(f"✓ Session ID: {session_id}...")
    
    # Retrieve and display memory context
    try:
        context = await memory.get_context()
        if context and context.strip():
            context_preview = context[:300] + "..." if len(context) > 300 else context
            print(f"\n📚 Memory context loaded ({len(context)} chars):")
            for line in context_preview.split('\n')[:4]:
                if line.strip():
                    print(f"   {line[:75]}")
            print(f"   [... context continues ...]")
        else:
            print(f"\n📚 No previous memory - First session!")
    except Exception as e:
        print(f"⚠️  Could not load context: {type(e).__name__}")
    
    # Process conversation turns
    print(f"\n{'-'*80}")
    print("CONVERSATION:")
    print(f"{'-'*80}")
    
    for i, query in enumerate(queries, 1):
        print(f"\nTurn {i}:")
        print(f"👤 User: {query}")
        
        try:
            # Run agent - memory injection happens automatically here!
            response = await agent.run(query)
            
            # Extract response text safely
            response_text = str(response)
            if hasattr(response, 'text'):
                response_text = response.text
            elif hasattr(response, 'content'):
                response_text = response.content
            
            # Show response (truncated for display)
            display_text = response_text[:250] + "..." if len(response_text) > 250 else response_text
            print(f"\n🤖 Advisor: {display_text}")
            
        except Exception as e:
            print(f"❌ Error in turn {i}: {type(e).__name__}: {str(e)[:100]}")
            raise
    
    # End session - triggers reflection and insight extraction
    print(f"\n{'-'*80}")
    try:
        result = await memory.end_session()
        insights_count = 0
        
        if isinstance(result, dict) and 'insights_extracted' in result:
            insights_count = len(result.get('insights_extracted', []))
        
        print(f"✓ Session ended")
        print(f"   • Conversation turns: {len(queries)}")
        print(f"   • 💡 Insights extracted: {insights_count}")
        print(f"   • (Profile updated automatically)")
        
    except Exception as e:
        print(f"⚠️  Error ending session: {type(e).__name__}")

# Run all three sessions
async def execute_all_sessions():
    """Execute the complete 3-session demo."""
    
    # SESSION 1: Initial Consultation
    await run_session(
        agent=agent,
        memory=memory,
        session_name="Session 1: Initial Consultation (Build Profile)",
        queries=[
            "Hi! I'm Sarah, 35 years old, software engineer making $150,000/year.",
            "I'm comfortable with moderate-to-high risk since I have 30 years until retirement.",
            "My employer offers a 401k with 4% match. What's the best strategy?"
        ]
    )
    
    # SESSION 2: Investment Strategy (Agent recalls context)
    await run_session(
        agent=agent,
        memory=memory,
        session_name="Session 2: Investment Strategy (Recall Context)",
        queries=[
            "Based on what we discussed before, what asset allocation do you recommend?",
            "Should I include international stocks given my risk tolerance?",
        ]
    )
    
    # SESSION 3: Tax Planning (Agent uses accumulated context)
    await run_session(
        agent=agent,
        memory=memory,
        session_name="Session 3: Tax Planning (Use Accumulated Knowledge)",
        queries=[
            "Given my income and the retirement accounts we discussed, how can I optimize taxes?",
            "Should I consider Roth conversions based on my profile?",
        ]
    )
    
    print(f"\n{'='*80}")
    print("✅ ALL SESSIONS COMPLETE")
    print(f"{'='*80}")
    print("\n📊 Summary:")
    print("   • Session 1: Built Sarah's profile (age, income, horizon, risk tolerance)")
    print("   • Session 2: Agent automatically recalled Session 1 context")
    print("   • Session 3: Agent used knowledge from Sessions 1 & 2 for tax advice")
    print("\n💡 Key insight: ALL memory management was AUTOMATIC!")
    print("   No manual add_turn(), store_response(), or get_context() calls needed.")

# Execute the demo
print("\n⏳ Starting 3-session demo...\n")
try:
    await execute_all_sessions()
except Exception as e:
    print(f"\n❌ Error during sessions: {type(e).__name__}")
    print(f"   {str(e)[:200]}")
    import traceback
    traceback.print_exc()


MULTI-SESSION DEMO: Automatic Memory Management in Action

⏳ Starting 3-session demo...


SESSION: Session 1: Initial Consultation (Build Profile)
Loaded sqlite-vec via Python package
[Orchestrator] Starting session 46042bb3-c435-468a-9089-ce9e7c6d0035
[MemoryKeeper] Initializing session context for user: sarah_demo
  ℹ No long-term insight found for user (will be created after sufficient sessions)
  ✓ Loaded 0 recent session summaries
✓ Session ID: 46042bb3...

📚 Memory context loaded (51 chars):
   <session_initialization>
   </session_initialization>
   [... context continues ...]

--------------------------------------------------------------------------------
CONVERSATION:
--------------------------------------------------------------------------------

Turn 1:
👤 User: Hi! I'm Sarah, 35 years old, software engineer making $150,000/year.
[MemoryKeeper] Added user turn. Buffer: 1/6
[MemoryKeeper] Added assistant turn. Buffer: 2/6

🤖 Advisor: Nice to meet you, Sarah. Thanks for shar

## Step 9-10: Analyze Results & Cleanup

### What We're Checking

1. **Memory Search** - Can the system find information semantically?
   - Query: "What is Sarah's risk tolerance?"
   - Should return relevant info from Sessions 1-3

2. **Sessions List** - All three sessions recorded?
   - Should show 3 sessions with summaries

3. **Insights Extracted** - Did system learn about Sarah?
   - Demographics (age, income, role)
   - Risk tolerance & preferences
   - Investment strategy & goals
   - Tax considerations

### Key Learnings

✅ **Automatic Context Injection**: Agent Framework's `before_run()` hook loads memory  
✅ **Automatic Turn Capture**: Agent Framework's `after_run()` hook stores responses  
✅ **Multi-Session Learning**: Agent improves with each conversation  
✅ **Insight Extraction**: System automatically identifies key facts  
✅ **Zero Manual Code**: Developer doesn't manage lifecycle - it's automatic!  

In [3]:
# ============================================================================
# STEP 9-10: Inspect Memory & Cleanup
# ============================================================================

print("\n" + "="*80)
print("MEMORY INSPECTION & ANALYSIS")
print("="*80)

# Inspect stored memory
async def inspect_and_cleanup():
    """Inspect memory contents and clean up."""
    
    try:
        # Start inspection session (required for database access)
        await memory.start_session()
        
        # ===== Test Memory Search =====
        print("\n🔍 MEMORY SEARCH TEST")
        print("-" * 80)
        print("Query: 'What is Sarah's risk tolerance?'")
        result = await memory.search("What is Sarah's risk tolerance?")
        if result:
            print(f"\n✅ Found relevant information:")
            print(f"{result[:300]}...")
        else:
            print("ℹ️  No search results (embeddings may not be configured)")
        
        # ===== Show Sessions =====
        print("\n📅 SESSIONS RECORDED")
        print("-" * 80)
        sessions = await memory.get_sessions()
        print(f"Total sessions: {len(sessions)}\n")
        for i, s in enumerate(sessions, 1):
            summary = s.get('summary', '')[:100]
            print(f"{i}. {summary}...")
        
        # ===== Show Insights =====
        print("\n💡 EXTRACTED INSIGHTS")
        print("-" * 80)
        insights = await memory.get_insights()
        print(f"Total insights: {len(insights)}\n")
        
        for i, insight in enumerate(insights[:10], 1):
            if isinstance(insight, dict):
                text = insight.get('insight_text', insight.get('insight', str(insight)))
            else:
                text = str(insight)
            
            category = insight.get('category', 'unknown') if isinstance(insight, dict) else 'unknown'
            confidence = insight.get('confidence', 0) if isinstance(insight, dict) else 0
            
            print(f"{i}. [{category}] {text[:80]}...")
            if confidence:
                print(f"   Confidence: {confidence:.0%}")
        
        await memory.end_session()
        
    except Exception as e:
        print(f"⚠️  Error during inspection: {type(e).__name__}: {str(e)[:100]}")

# Run inspection
print("\n⏳ Inspecting memory...")
try:
    await inspect_and_cleanup()
except Exception as e:
    print(f"Inspection error: {e}")

# Close memory connection
print("\n🧹 CLEANUP")
print("-" * 80)

print("Closing memory connection...")
try:
    await memory.close()
    print("✅ Memory connection closed")
except Exception as e:
    print(f"⚠️  Error: {e}")

# Clean up database
print(f"\nDeleting database: {DB_PATH}")
for attempt in range(5):
    try:
        if os.path.exists(DB_PATH):
            os.remove(DB_PATH)
            print("✅ Database deleted")
        break
    except PermissionError:
        if attempt < 4:
            time.sleep(0.5)
        else:
            print("⚠️  Could not delete (file in use)")

# Summary
print("\n" + "="*80)
print("📚 KEY LEARNINGS: AUTOMATIC MEMORY MANAGEMENT")
print("="*80)
print("""
✅ What Happened:
   Session 1: Agent met Sarah, learned about her
   Session 2: Agent automatically recalled Session 1 details
   Session 3: Agent combined knowledge from Sessions 1+2

✅ How It Worked (No Manual Code!):
   1. Pass memory to Agent as: context_providers=[memory]
   2. Agent Framework's before_run() automatically calls memory.before_run()
   3. Memory injects context from previous sessions
   4. Agent responds
   5. Agent Framework's after_run() automatically calls memory.after_run()
   6. Response stored - no developer code needed!

✅ What Makes This Powerful:
   ✓ Zero manual memory management
   ✓ Agent naturally references past conversations
   ✓ User profile builds over time
   ✓ System extracts insights automatically
   ✓ Works with SQLite (local) or cloud backends
   ✓ Scales to millions of users

✅ Real-World Applications:
   • Customer support AI (remembers previous issues)
   • Financial advisor (learns user preferences)
   • Health coach (tracks progress)
   • Therapist bot (builds patient understanding)
   • Tutoring system (adapts to student level)
   • HR assistant (knows employee history)

✅ Technical Flow:
   Agent.run(query)
   ├─ Calls before_run() [Memory injection]
   │  └─ memory.get_context() loads previous context
   ├─ Agent processes query with context
   ├─ Agent generates response
   └─ Calls after_run() [Memory storage]
      └─ memory.add_turn() stores response
        
   All automatic - developer only needs to pass context_providers=[memory]

""")
print("="*80)
print("✅ DEMO COMPLETE - Automatic memory management validated!")
print("="*80)


MEMORY INSPECTION & ANALYSIS

⏳ Inspecting memory...
[Orchestrator] Starting session 781b67eb-5fa1-468d-82b5-06915e071e2d
[MemoryKeeper] Initializing session context for user: sarah_demo
  ✓ Loaded long-term insight profile (3469 chars)
     Preview: USER: sarah_demo

SUMMARY
Sarah is a 35-year-old software engineer with strong earning power, a long investment horizon, and high equity risk toleranc...
     📋 Session 9eb27d53...: The user asked whether Roth conversions make sense given their profile as a 35-year-old high-earning...
     📋 Session 0d47c767...: The user asked for a recommended asset allocation and clarification on whether to include internatio...
     📋 Session 46042bb3...: The user (Sarah, 35-year-old software engineer earning $150,000) began a retirement planning convers...
  ✓ Loaded 3 recent session summaries

🔍 MEMORY SEARCH TEST
--------------------------------------------------------------------------------
Query: 'What is Sarah's risk tolerance?'

✅ Found relevan